In [ ]:
import pandas as pd
import warnings

from skforecast.exceptions import IgnoredArgumentWarning

import seaborn as sns
sns.set_theme(palette="colorblind")

warnings.simplefilter('ignore', category=IgnoredArgumentWarning)

# Ex 6.1

Create three SES models on the Algeria export dataset, with alpha=0.2, 0.5, 0.8. Generate 10-steps ahead forecasts for each model. Plot the fitted and forecast values. Describe how the fitted curve and the forecast values depend on alpha.

In [ ]:
from skforecast.stats._ets import Ets

df = pd.read_csv("algeria_exports.csv", parse_dates=["ds"], index_col="ds", usecols=["ds", "y"])

# explicitly set the frequency of the index to yearly
df.index.freq = 'YE'

In [ ]:
for a in [0.2, 0.5, 0.8]:
    
    ???

    # plot original series, SES-fitted values and predictions
    tdf = pd.DataFrame({"y": df['y'], "fitted": fitted})
    tdf = tdf.merge(predictions, left_index=True, right_index=True, how='outer')
    tdf.plot(figsize=(12,4), title=f"SES with alpha={a}")

Comments:

???

# Ex.6.2

The Australian tourism dataset ([download link](https://otexts.com/fpppy/data/tourism.csv)) contains quarterly numbers of visitor nights during trips made with different purposes. Calculate the values for nation-wide trips, where the purpose was "Holiday" and plot the series. Is there a seasonality? If yes, what is the length of the seasonal cycle?

Construct an Exponential Smoothing model using the AutoETS method on the series, specifying the appropriate length of the seasonal pattern, and describe the best settings found for $\alpha$, $\beta$, $\gamma$ and $\phi$. 

Generate forecasts for the next 10 years. Comment on the produced forecasts.

In [ ]:
df = pd.read_csv(???)

# sum the values across different states for the "Holiday" purpose
df = df[df["Purpose"] == "???"].groupby("ds")["y"].sum().to_frame()

# explicitly set the frequency of the index to quarterly
df.index.freq = 'QS-OCT'

df.info()

In [ ]:
# plot the calculated series
???

In [ ]:
# Run AutoETS with an appropriate value of m for the data
???

Comments:

* alpha=???
* beta=???
* gamma=???
* phi=???

In [ ]:
# values fitted on the full dataset
fitted = model.fitted_values_

# predicted values at the end of the dataset
predictions = pd.DataFrame({"yhat": ???}, 
                           index=pd.date_range(start=df.index[-1], periods=???, freq='QE'))

# plot original series, AutoETS-fitted values and predictions
???

Comments:

???

In [ ]:
# Optional step: re-fitting the model with a dampening trend
model = Ets(m=4, model=None, damped=True)

???

# plot original series, AutoETS-fitted values and predictions
tdf = pd.DataFrame({"y": df['y'], "fitted": fitted})
tdf = tdf.merge(predictions, left_index=True, right_index=True, how='outer')
tdf.plot(figsize=(12,4))

# Ex 6.3

Using the Holiday series from the previous exercise and the best model, run a backtesting of the same model using one-step ahead forecasts, 80% series for model training and compare its quality with a last-value baseline. 

Evaluate the models using RMSE and MAE. 

Test the significance of the differences between the two models using Diebold-Mariano test. What is the conclusion?

In [ ]:
from skforecast.model_selection import TimeSeriesFold
from skforecast.model_selection import backtesting_forecaster
from sklearn.metrics import root_mean_squared_error

cv = TimeSeriesFold(
         ???
     )

## Last-value baseline

In [ ]:
from skforecast.recursive import ForecasterEquivalentDate

last_value_forecaster = ForecasterEquivalentDate(offset=1, n_offsets=1)

results, predictions_lvbaseline = backtesting_forecaster(
                                        ???
                               )
results

## AutoETS

In [ ]:
from skforecast.stats._ets import Ets
from skforecast.recursive import ForecasterStats
from skforecast.model_selection._validation import backtesting_stats

ets_forecaster = ForecasterStats(estimator=Ets(m=4, model=None))

results, predictions_ets = backtesting_stats(
                                        ???
                               )
results

In [ ]:
# Run Diebold-Mariano test to compare the forecasts of the two models

from dm_test import dm_test

true_values = df.iloc[64:,0]
forecast1 = predictions_lvbaseline['pred']
forecast2 = predictions_ets['pred']

???